# Step 1: Import Required Libraries

In [18]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from scipy.stats import randint
from sklearn.model_selection import cross_val_score

# Step 2: Load the Dataset

In [4]:
# Load Wine dataset
wine = load_wine()

# Features and target
X = wine.data
y = wine.target

# Convert to DataFrame
df = pd.DataFrame(X, columns=wine.feature_names)
df["target"] = y

print(df.head())
print(df.shape)

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  target  
0          

# Step 3: Data Preprocessing

In [5]:
print(df.isnull().sum())

alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
target                          0
dtype: int64


# Step 4: Build Random Forest Model

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

# Step 5: Evaluate Initial Model

In [20]:
from sklearn.model_selection import cross_val_score

default_cv = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Default Random Forest CV Scores:", default_cv)
print("Mean CV Score:", default_cv.mean())

Default Random Forest CV Scores: [1.         0.93103448 1.         1.         1.        ]
Mean CV Score: 0.9862068965517242


# Step 6: Hyperparameter Tuning using GridSearchCV

In [10]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [11]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV Score:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Best CV Score:
0.9862068965517242


# Evaluate Best Grid Search Model

In [21]:
grid_cv = cross_val_score(
    grid_search.best_estimator_,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Grid Search CV Scores:", grid_cv)
print("Mean CV Score:", grid_cv.mean())

Grid Search CV Scores: [1.         0.93103448 1.         1.         1.        ]
Mean CV Score: 0.9862068965517242


# Step 7: Hyperparameter Tuning using RandomizedSearchCV

# Define Parameter Distribution

In [13]:
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'bootstrap': [True, False]
}

# Perform Random Search

In [14]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    random_state=42,
    scoring='accuracy',
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV Score:")
print(random_search.best_score_)

Best Parameters:
{'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 6, 'n_estimators': 51}
Best CV Score:
0.993103448275862


# Evaluate Random Search Model

In [22]:
random_cv = cross_val_score(
    random_search.best_estimator_,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Random Search CV Scores:", random_cv)
print("Mean CV Score:", random_cv.mean())

Random Search CV Scores: [1.         0.96551724 1.         1.         1.        ]
Mean CV Score: 0.993103448275862


# Step 8: Compare Results

In [19]:
print("Default RF CV Score:", cross_val_score(
    rf, X_train, y_train, cv=5).mean())

print("GridSearch Best CV Score:", grid_search.best_score_)

print("RandomSearch Best CV Score:", random_search.best_score_)

Default RF CV Score: 0.9862068965517242
GridSearch Best CV Score: 0.9862068965517242
RandomSearch Best CV Score: 0.993103448275862


# Step 9: Feature Importance

In [23]:
importance = pd.DataFrame({
    'Feature': wine.feature_names,
    'Importance': best_grid_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

                         Feature  Importance
6                     flavanoids    0.198375
9                color_intensity    0.172235
12                       proline    0.144254
0                        alcohol    0.121256
11  od280/od315_of_diluted_wines    0.096693
10                           hue    0.084098
4                      magnesium    0.046284
5                  total_phenols    0.037123
1                     malic_acid    0.034526
3              alcalinity_of_ash    0.027121
8                proanthocyanins    0.016166
2                            ash    0.013918
7           nonflavanoid_phenols    0.007952
